# Notebook 03: Empirical Data Collection Pipeline
This notebook implements the data collection scripts for pulling Google Trends, GDELT media salience, PDK/EdNext polls, and state-level ESSA accountability plans.


### Important Project Safety Notice

Before executing or citing the findings in this notebook, please read the public guidance on what this project is and is not claiming:  

[docs/not_saying.md](../docs/not_saying.md) - *What This Theory Is NOT Claiming*


## 1. Library Imports
We load dependencies. Make sure you run `00_setup.ipynb` first to verify package installations.


In [ ]:
import os
import time
import pandas as pd
import numpy as np
import urllib3.util
from dotenv import load_dotenv

# Load environment variables from .env file (looks in parent directories as well)
load_dotenv()

# Monkey-patch urllib3 Retry constructor to prevent pytrends method_whitelist crash in urllib3 >= 2.0
if not getattr(urllib3.util.Retry, '_is_patched', False):
    original_init = urllib3.util.Retry.__init__
    def patched_init(self, *args, **kwargs):
        if 'method_whitelist' in kwargs:
            kwargs['allowed_methods'] = kwargs.pop('method_whitelist')
        original_init(self, *args, **kwargs)
    urllib3.util.Retry.__init__ = patched_init
    urllib3.util.Retry._is_patched = True

from pytrends.request import TrendReq
from google.cloud import bigquery
from google.auth.exceptions import DefaultCredentialsError

os.makedirs('../data/raw', exist_ok=True)
print('Imports, load_dotenv, and monkey-patch successful. data/raw folder verified.')


## 2. Google Trends (Pytrends) Panel Collection
Because Google Trends normalizes search volume relative to the peak region/time of each query (0–100 index), raw queries across different states are not directly comparable.
### Normalization Methodology
1. Fetch the national regional index (`interest_by_region`) for the keyword to determine cross-sectional scaling factors.
2. Fetch the state-specific time series (`interest_over_time`).
3. Multiply the state's time series by its regional scale factor to build a globally comparable state-year panel.
### Rate-Limit Protection
Includes exponential backoff retries when hitting HTTP 429 errors.


In [ ]:
def fetch_trends_panel(keywords, start_year=2010, end_year=2024, test_mode=False):
    # hl=en-US, tz=360 (Central Standard Time)
    pytrends = TrendReq(hl='en-US', tz=360, retries=5, backoff_factor=3)
    timeframe = f'{start_year}-01-01 {end_year}-12-31'
    
    panel_data = []
    
    # 50 states list
    states = [
        'US-AL', 'US-AK', 'US-AZ', 'US-AR', 'US-CA', 'US-CO', 'US-CT', 'US-DE', 'US-FL', 'US-GA',
        'US-HI', 'US-ID', 'US-IL', 'US-IN', 'US-IA', 'US-KS', 'US-KY', 'US-LA', 'US-ME', 'US-MD',
        'US-MA', 'US-MI', 'US-MN', 'US-MS', 'US-MO', 'US-MT', 'US-NE', 'US-NV', 'US-NH', 'US-NJ',
        'US-NM', 'US-NY', 'US-NC', 'US-ND', 'US-OH', 'US-OK', 'US-OR', 'US-PA', 'US-RI', 'US-SC',
        'US-SD', 'US-TN', 'US-TX', 'US-UT', 'US-VT', 'US-VA', 'US-WA', 'US-WV', 'US-WI', 'US-WY'
    ]
    
    if test_mode:
        print('Running in Test Mode (2 states, 1 keyword only)...')
        states = ['US-CA', 'US-NY']
        keywords = [keywords[0]]
        
    for kw in keywords:
        print(f'Fetching cross-sectional scaling factors for "{kw}"...')
        time.sleep(2)
        try:
            pytrends.build_payload([kw], timeframe=timeframe, geo='US')
            region_df = pytrends.interest_by_region(resolution='REGION', inc_low_vol=True, inc_geo_code=True)
            scaling_factors = region_df.set_index('geoCode')[kw].to_dict()
        except Exception as e:
            print(f'Error fetching regional interest: {e}. Defaulting to uniform weights.')
            scaling_factors = {s: 50.0 for s in states}
            
        print(f'Fetching state-specific time series for "{kw}"...')
        for state in states:
            scale = scaling_factors.get(state, 50.0) / 100.0
            print(f'  Querying {state} (Scale factor: {scale:.3f})...')
            
            attempts = 0
            success = False
            while attempts < 3 and not success:
                try:
                    time.sleep(1.5)  # Throttle request frequency
                    pytrends.build_payload([kw], timeframe=timeframe, geo=state)
                    ts_df = pytrends.interest_over_time()
                    
                    if not ts_df.empty:
                        # Resample monthly data to annual mean
                        annual_avg = ts_df[kw].resample('YE').mean()
                        for date, val in annual_avg.items():
                            panel_data.append({
                                'state': state,
                                'year': date.year,
                                'keyword': kw,
                                'raw_svi': val,
                                'scaled_svi': val * scale
                            })
                        success = True
                    else:
                        success = True  # Empty response
                except Exception as e:
                    wait_time = (attempts + 1) * 10
                    print(f'    Rate limit or error on {state}. Waiting {wait_time}s before retry...')
                    time.sleep(wait_time)
                    attempts += 1
                    
    return pd.DataFrame(panel_data)


### Test Run: Fetching Google Trends Sample
Run this test block to verify connection and data structure. It queries only California and New York for the term `'Common Core'`.


In [ ]:
df_test = fetch_trends_panel(
    keywords=['Common Core', 'standardized testing', 'opt out testing', 'highway funding'],
    start_year=2015,
    end_year=2018,
    test_mode=True
)
print(df_test.head(10))


### Full Run: Fetching 50-State Trends Panel (Commented out to prevent rate-blocking)
To build the complete panel, uncomment the cell below and run it. This will take ~5–10 minutes to complete.


In [ ]:
# df_trends = fetch_trends_panel(
#     keywords=['Common Core', 'standardized testing', 'opt out testing', 'highway funding'],
#     start_year=2010,
#     end_year=2024,
#     test_mode=False
# )
# df_trends.to_csv('../data/raw/google_trends_raw.csv', index=False)
# print('Complete Trends panel saved to data/raw/google_trends_raw.csv')


## 3. GDELT Media Salience BigQuery Extraction
This section pulls daily state-level article counts matching education keywords and highway funding keywords.
If you do not have local Google Cloud credentials configured, the script will output the exact SQL string for you to copy-paste into the BigQuery console.


In [ ]:
def run_gdelt_query():
    query = '''
    SELECT
      EXTRACT(YEAR FROM PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING))) as year,
      ActionGeo_ADM1Code as state_code,
      COUNT(1) as total_events,
      SUM(CASE WHEN (Actor1Name LIKE '%EDUCATION%' OR Actor2Name LIKE '%EDUCATION%') THEN 1 ELSE 0 END) as edu_events,
      SUM(CASE WHEN (Actor1Name LIKE '%HIGHWAY%' OR Actor2Name LIKE '%HIGHWAY%') THEN 1 ELSE 0 END) as highway_events
    FROM `gdelt-bq.full.events`
    WHERE ActionGeo_CountryCode = 'US'
      AND SQLDATE >= 20100101
      AND SQLDATE <= 20241231
    GROUP BY year, state_code
    ORDER BY year, state_code;
    '''
    
    try:
        client = bigquery.Client()
        print('GCP credentials found. Running BigQuery query...')
        query_job = client.query(query)
        df_gdelt = query_job.to_dataframe()
        df_gdelt.to_csv('../data/raw/gdelt_media_salience.csv', index=False)
        print('GDELT query completed and saved to data/raw/gdelt_media_salience.csv')
    except DefaultCredentialsError:
        print('No Google Cloud Credentials found. Please run the following query manually in the BigQuery console:')
        print('URL: https://console.cloud.google.com/bigquery')
        print('-' * 60)
        print(query)
        print('-' * 60)
        print('After downloading, save the CSV to "data/raw/gdelt_media_salience.csv"')

run_gdelt_query()


## 4. Public Polls & ANES Download Stubs
Pulls state-level ANES aggregates and PDK/EdNext polls if available, creating a standardized stub format.


In [ ]:
def download_poll_data():
    # Representative PDK/EdNext time-series aggregates stub
    polls_stub = pd.DataFrame([
        {'year': 2010, 'keyword': 'standardized testing', 'national_support': 0.58, 'national_oppose': 0.35},
        {'year': 2012, 'keyword': 'standardized testing', 'national_support': 0.52, 'national_oppose': 0.42},
        {'year': 2014, 'keyword': 'standardized testing', 'national_support': 0.44, 'national_oppose': 0.51},
        {'year': 2016, 'keyword': 'standardized testing', 'national_support': 0.40, 'national_oppose': 0.56},
        {'year': 2018, 'keyword': 'standardized testing', 'national_support': 0.43, 'national_oppose': 0.52},
        {'year': 2020, 'keyword': 'standardized testing', 'national_support': 0.46, 'national_oppose': 0.48},
        {'year': 2022, 'keyword': 'standardized testing', 'national_support': 0.38, 'national_oppose': 0.55},
        {'year': 2024, 'keyword': 'standardized testing', 'national_support': 0.35, 'national_oppose': 0.58}
    ])
    
    polls_stub.to_csv('../data/raw/public_polls_raw.csv', index=False)
    print('PDK/EdNext poll aggregates stub created at "data/raw/public_polls_raw.csv"')

download_poll_data()


## 5. ESSA Consolidated State Plan PDF Coding Template
Generates a structured coding sheet in `data/raw/essa_plan_coding_sheet.csv` where you can manually record accountability weights from state PDFs.


In [ ]:
def generate_essa_coding_sheet():
    states = [
        'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA',
        'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
        'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
        'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
        'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 'DC'
    ]
    
    filepath = '../data/raw/essa_plan_coding_sheet.csv'
    if os.path.exists(filepath):
        print(f'ESSA coding sheet already exists at "{filepath}". Skipping to avoid overwriting.')
        return
        
    df = pd.DataFrame({
        'state': states,
        'academic_achievement_weight': [np.nan] * len(states),
        'academic_growth_weight': [np.nan] * len(states),
        'school_quality_success_weight': [np.nan] * len(states),
        'ell_progress_weight': [np.nan] * len(states),
        'status': ['Not Started'] * len(states),
        'notes': [''] * len(states)
    })
    
    df.to_csv(filepath, index=False)
    print(f'Generated new ESSA coding sheet template at "{filepath}"')

generate_essa_coding_sheet()


### Codebook Guide for State ESSA Plan PDF Coding
To find these values in the state plans:
1. Download the state's ESSA Consolidated State Plan PDF (available on the U.S. Dept of Education website).
2. Search for the section labeled **Title I, Part A: Academic Assessments** or **Indicators and Weights** (typically Section 4).
3. Look for the weight distribution table summarizing percentages of total school accountability ratings.
4. Record the weights in the CSV coding sheet. Make sure weights across columns sum to 100% (or the state's total index points scale).
